# k-Nearest Neighbors

## > Install Dependencies

In [ ]:
%pip install scikit-learn

import numpy as np
import pandas as pd
import hashlib
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


## > Import Parameters and Dataset

In [ ]:

student_df = pd.read_csv("student_dataset.csv")
def add_username_to_plot(username="gracy.garg"):
    """
    Adds the username to the top-right corner of the current plot.
    """
    plt.text(
        0.95, 0.95, "gracy.garg",
        ha='right', va='top',
        transform=plt.gca().transAxes,
        fontsize=10, color='gray', alpha=0.7
    )
username = "gracy.garg"
seed = int ( hashlib . sha256 ( username . encode () ) . hexdigest () , 16)
seed = seed % (2**32)


## > Helper class for feature transformations

In [ ]:
class PerFeatureTransformer:
    def __init__(self):
        """Initializes the PerFeatureTransformer with empty dictionaries for transformers and fitted objects."""
        self.transformers = {}
        self.fitted = {}

    def fit(self, df: pd.DataFrame, params: dict[str, str]) -> None:
        """Fits transformers for each feature based on the given types in params."""
        for feature, ttype in params.items():
            if ttype == "standard":
                scaler = StandardScaler()
                scaler.fit(df[[feature]])
                self.fitted[feature] = (ttype, scaler)
            elif ttype == "ordinal":
                enc = OrdinalEncoder()
                enc.fit(df[[feature]])
                self.fitted[feature] = (ttype, enc)
            elif ttype == "onehot":
                enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore") # Handle sparse matrix from OneHotEncoder
                enc.fit(df[[feature]])
                self.fitted[feature] = (ttype, enc)

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        """Applies the fitted transformers to the corresponding features and returns a NumPy array."""
        outputs = []
        for feature, (ttype, transformer) in self.fitted.items():
            X_feat = df[[feature]]
            transformed = transformer.transform(X_feat)
            # Ensure the output is a 2D array
            if transformed.ndim == 1:
                transformed = transformed.reshape(-1, 1)
            
            outputs.append(transformed)
            
        return np.hstack(outputs)

    def fit_transform(self, df: pd.DataFrame, params: dict[str, str]) -> np.ndarray:
        """Fits and transforms all features in one step using the given transformation parameters"""
        self.fit(df, params)
        return self.transform(df)

## > Predicting gender using KNN

In [ ]:

class KNNGenderPredictor:
    def __init__(self, student_df: pd.DataFrame, username: str):
        """Initializes the KNNGenderPredictor with the full student dataset."""
        self.df = student_df
        self.username = username
        self.transformer = None
        self.params = None
        self.train_df = self.val_df = self.test_df = None

    def train_val_test_split(self, test_size: float = 0.2, val_size: float = 0.2, seed: int = seed):
        """Splits the dataset into train, validation, and test sets."""
        train_val, self.test_df = train_test_split(self.df, test_size=test_size, random_state=seed)
        rel_val_size = val_size / (1 - test_size)  
        self.train_df, self.val_df = train_test_split(train_val, test_size=rel_val_size, random_state=seed)
        return self.train_df, self.val_df, self.test_df

    def get_feature_matrix_and_labels(self, df: pd.DataFrame, features: list[str]):
        """Returns the feature matrix and labels for the given DataFrame and features."""
        X = self.transformer.transform(df[features])
        y = df["gender"].values
        return X, y
    
    def get_knn_accuracy_vs_k(self, k_values: list[int], distance: str = "euclidean"):
        """ Calculates accuracy scores for various k values on the validation set . """
        X_train, y_train = self.get_feature_matrix_and_labels(self.train_df, list(self.params.keys()))
        X_val, y_val = self.get_feature_matrix_and_labels(self.val_df, list(self.params.keys()))
        accuracies = []
        for k in k_values:
            knn = KNeighborsClassifier(n_neighbors=k, metric=distance)
            knn.fit(X_train, y_train)
            preds = knn.predict(X_val)
            acc = accuracy_score(y_val, preds)
            # print(f"k={k}: accuracy={acc:.4f}")
            accuracies.append(acc)
        return accuracies


## > Analysis Functions

In [ ]:
class KNNAnalysis(KNNGenderPredictor):
    def plot_knn_accuracy_vs_k(self, k_values: list[int], distance: str = "euclidean"):
        """Function to plot accuracy vs k for a given distance metric."""
        color_map = {"euclidean": "blue", "manhattan": "orange", "cosine": "green"}
        color = color_map.get(distance, "black")
        accuracies = self.get_knn_accuracy_vs_k(k_values, distance)
        plt.plot(k_values, accuracies, marker="o", color=color)
        max_idx = np.argmax(accuracies)
        max_k = k_values[max_idx]
        max_acc = accuracies[max_idx]
        plt.scatter(max_k, max_acc, color='red', s=100, edgecolor='black', zorder=5)
        plt.annotate(
            f"Max: {max_acc:.3f}\n(k={max_k})",
            (max_k, max_acc),
            textcoords="offset points",
            xytext=(0, 15),
            ha='center',
            fontsize=10,
            color='red',
        )
        plt.title(f"KNN Accuracy vs k ({distance})")
        plt.xlabel("k")
        plt.ylabel("Validation Accuracy")
        add_username_to_plot(self.username)
        plt.show()

    def get_knn_f1_heatmap(self, k_values: list[int], distances: list[str]):
        """Function to compute F1 scores for various k values and distance metrics, returning a DataFrame."""
        X_train, y_train = self.get_feature_matrix_and_labels(self.train_df, list(self.params.keys()))
        X_val, y_val = self.get_feature_matrix_and_labels(self.val_df, list(self.params.keys()))
        results = {}
        for dist in distances:
            row = []
            for k in k_values:
                knn = KNeighborsClassifier(n_neighbors=k, metric=dist)
                knn.fit(X_train, y_train)
                preds = knn.predict(X_val)
                row.append(f1_score(y_val, preds, average="macro"))
            results[dist] = row

        f1_df = pd.DataFrame(results, index=k_values)
        feature_map = {"euclidean": "Euclidean", "manhattan": "Manhattan", "cosine": "Cosine"}
        f1_df.rename(columns=feature_map, inplace=True)
        print("F1 Scores Table (k vs Distance):")
        display(f1_df.style.set_properties(**{'font-size': '16px'}))
        return f1_df

    def plot_knn_f1_heatmap(self, f1_scores_df: pd.DataFrame):
        """Function to plot a heatmap of F1 scores."""
        plt.figure(figsize=(10, 6))
        sns.heatmap(f1_scores_df.T, annot=True, cmap="coolwarm", cbar_kws={"label": "F1 Score"})        
        plt.title("F1 Score Heatmap (k vs Distance)")
        plt.xlabel("k")
        plt.ylabel("Distance Metric")
        add_username_to_plot(self.username)
        plt.show()

    def get_knn_f1_single_feature_table(self, k_values: list[int], features: list[str], distance: str = "euclidean"):
        """Function to compute F1 scores for KNN using a single feature across various k values, returning a DataFrame."""
        _, y_train = self.get_feature_matrix_and_labels(self.train_df, list(self.params.keys()))
        _, y_test = self.get_feature_matrix_and_labels(self.test_df, list(self.params.keys()))
        results = {}
        for feature in features:
            f1s = []
            t = PerFeatureTransformer()
            single_param = {feature: self.params[feature]}
            t.fit(self.train_df, single_param)
            X_train = t.transform(self.train_df[[feature]])
            X_test = t.transform(self.test_df[[feature]])
            for k in k_values:
                knn = KNeighborsClassifier(n_neighbors=k, metric=distance)
                knn.fit(X_train, y_train)
                preds = knn.predict(X_test)
                f1s.append(f1_score(y_test, preds, average="macro"))
            results[feature] = f1s
        f1_df = pd.DataFrame(results, index=k_values)
        feature_map = {"gpa": "GPA", "major": "Major", "program": "Program"}
        f1_df.rename(columns=feature_map, inplace=True)
        print("F1 Score Table for KNN using Single Feature (k vs Feature):")
        display(f1_df.style.set_properties(**{'font-size': '16px'}))
        return f1_df

## > Split the dataset and apply the data transforms.

In [ ]:
features = ["gpa", "major", "program"] 
params = {"gpa": "standard", "major": "ordinal", "program": "onehot"}

model = KNNGenderPredictor(student_df, "username")
train_df, val_df, test_df = model.train_val_test_split()
model.params = params
model.transformer = PerFeatureTransformer()
model.transformer.fit(train_df, params)

## > Predicting gender using KNN.

## a) With Euclidean distance

In [ ]:
k_values = list(range(1, 22, 2))
analysis_model = KNNAnalysis(student_df, username="gracy.garg")
analysis_model.train_val_test_split()
analysis_model.params = params
analysis_model.transformer = PerFeatureTransformer()
analysis_model.transformer.fit(train_df, params)
analysis_model.plot_knn_accuracy_vs_k(k_values, distance="euclidean")

## b) Using Manhattan distance  

In [ ]:
analysis_model.plot_knn_accuracy_vs_k(k_values, distance="manhattan")

## c) Cosine Similarity

In [ ]:
analysis_model.plot_knn_accuracy_vs_k(k_values, distance="cosine")

## Compare

In [ ]:
def plot_knn_accuracy_vs_k_all(self, k_values: list[int], distances: list[str] = ["euclidean", "manhattan", "cosine"]):
    """Plot all KNN accuracy vs k for different distance metrics on the same graph to compare and analyse."""
    plt.figure(figsize=(10, 6))
    colors = ["tab:blue", "tab:orange", "tab:green"]
    for idx, dist in enumerate(distances):
        accuracies = self.get_knn_accuracy_vs_k(k_values, dist)
        plt.plot(k_values, accuracies, marker="o", color=colors[idx], label=f"{dist.capitalize()}")
        max_idx = np.argmax(accuracies)
        max_k = k_values[max_idx]
        max_acc = accuracies[max_idx]
        plt.scatter(max_k, max_acc, color=colors[idx], s=100, edgecolor='black', zorder=5)
        plt.annotate(
            f"Max: {max_acc:.3f}\n(k={max_k})",
            (max_k, max_acc),
            textcoords="offset points",
            xytext=(0, 15 + idx * 20), 
            ha='center',
            fontsize=10,
            color=colors[idx],
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=colors[idx], alpha=0.7)
        )
    plt.title("KNN Validation Accuracy vs k for Different Distance Metrics")
    plt.xlabel("k")
    plt.ylabel("Validation Accuracy")
    plt.legend(title="Distance Metric")
    add_username_to_plot(self.username)
    plt.tight_layout()
    plt.show()
    
plot_knn_accuracy_vs_k_all(analysis_model, k_values)

## > K × Distance function vs F-1 score.

In [ ]:
f1_scores = analysis_model.get_knn_f1_heatmap(k_values, ["euclidean", "manhattan", "cosine"])

**Observation** : F1 scores decrease as k increases: small k values overfit to noise, mid-range k balances bias–variance (~k=11), while large k underfit by oversmoothing toward the majority class. Euclidean and Manhattan show near-identical behavior, while Cosine occasionally outperforms but without consistent advantage.

In [ ]:
analysis_model.plot_knn_f1_heatmap(f1_scores)

## > Which distance metric performs better? Why?

* **Cosine similarity** achieved the highest overall validation accuracy (peaking at ~0.634 for k = 19), slightly outperforming Euclidean (0.626) and Manhattan (0.627). 
This is because Cosine distance measures the **angle between feature vectors rather than their raw magnitude**, making it particularly effective for datasets where features are a mix of categorical codes (e.g., major, program) and normalized numerical values (e.g., GPA). In such cases, the relative pattern of attributes is often more informative than absolute differences, allowing Cosine similarity to better group students with comparable academic profiles and thus improve accuracy.

* When considering F1 scores—which are more appropriate for imbalanced classification problems such as gender prediction **Euclidean and Manhattan distances** performed more consistently. Since they emphasize absolute differences in feature values, these metrics prevent the classifier from over-clustering around the majority class, leading to a **fairer balance between precision and recall**.

In summary, **Cosine similarity** offers slightly better raw accuracy due to its focus on feature alignment, while Euclidean and Manhattan distances provide more reliable performance in terms of balanced gender prediction.

## > Prediction based on Single Feature

In [ ]:
single_feature_results = analysis_model.get_knn_f1_single_feature_table(k_values, features, distance="euclidean")

## > Findings (Single Feature KNN – F1 Scores)

On comparing the predictive performance of KNN classifiers using one feature at a time (GPA, Major, Program) across different values of k, the F1 scores were evaluated on the test set for multiple distance metrics.

* GPA: Performance fluctuated around ~0.31–0.33. The best score was ~0.3278 at k=11. Overall, GPA as a single feature was moderately informative but less stable than Program.
* Major: The F1 score remained constant at 0.3031 for most k. This suggests that Major by itself has limited predictive power.
* Program: Consistently achieved the highest F1 score (~0.342–0.352) across different k, so Program is the most discriminative single feature. The best performance occurred at 
k=5,7,9,13 (~0.3517).

Overall:
Among the single features, Program provided the strongest predictive performance, followed by GPA, while Major contributed the least [Program>GPA>Major]. Increasing k beyond 9 did not improve the scores substantially and sometimes reduced performance.

## > Single-feature vs. Multi-feature Results

The single-feature F1 scores (GPA, Major, Program) are consistently lower than when all features are used together. Among individual features, Program performs the best (peaking around 0.35), while GPA fluctuates near 0.32 and Major remains stable at ~0.30. However, none of these outperform the multi-feature case, where combining attributes yields more reliable F1 scores (~0.33 across metrics).

This difference arises because a single feature provides only narrow distinguishing ability, forcing the model to base predictions on one aspect of a student’s profile. In contrast, using all features together creates a richer representation of similarity, enabling the distance metrics to capture more nuanced patterns. While “Program” alone is the strongest single predictor, it still lacks the consistency and balance achieved by leveraging multiple features.